In [1]:
import pandas as pd
import numpy as np

In [2]:
col_names = ["Sex","Length","Diameter","Height","Whole_weight",
             "Shucked_weight","Viscera_weight","Shell_weight","Rings"]

In [3]:
df = pd.read_csv("abalone.data", header=None, names=col_names)

In [4]:
df = pd.get_dummies(df, columns=["Sex"])

In [5]:
X = df.drop("Rings", axis=1).values.astype(float)
y = df["Rings"].values.reshape(-1, 1).astype(float)

In [6]:
X = (X - X.mean(axis=0)) / X.std(axis=0)

In [7]:
class AbaloneNet:
    def __init__(self, in_size, hid_size, out_size):
        self.w1 = np.random.randn(in_size, hid_size) * 0.01
        self.b1 = np.zeros((1, hid_size))
        self.w2 = np.random.randn(hid_size, out_size) * 0.01
        self.b2 = np.zeros((1, out_size))

    def sig(self, z):
        return 1 / (1 + np.exp(-z))

    def sig_grad(self, a):
        return a * (1 - a)

    def pass_forward(self, x):
        self.h_in = np.dot(x, self.w1) + self.b1
        self.h_out = self.sig(self.h_in)

        self.final_in = np.dot(self.h_out, self.w2) + self.b2
        self.final_out = self.final_in
        return self.final_out

    def pass_backward(self, x, y, out, lr):
        n = len(x)

        out_error = out - y
        dw2 = np.dot(self.h_out.T, out_error) / n
        db2 = np.sum(out_error, axis=0, keepdims=True) / n

        hid_error = np.dot(out_error, self.w2.T) * self.sig_grad(self.h_out)
        dw1 = np.dot(x.T, hid_error) / n
        db1 = np.sum(hid_error, axis=0, keepdims=True) / n

        self.w2 = self.w2 - lr * dw2
        self.b2 = self.b2 - lr * db2
        self.w1 = self.w1 - lr * dw1
        self.b1 = self.b1 - lr * db1

    def fit(self, x, y, epochs, lr):
        for i in range(epochs):
            out = self.pass_forward(x)
            self.pass_backward(x, y, out, lr)

            if (i + 1) % 100 == 0:
                err = np.mean((y - out) ** 2)
                print("epoch", i + 1, "error =", round(err, 4))

    def predict(self, x):
        return self.pass_forward(x)

In [8]:
model = AbaloneNet(X.shape[1], 7, 1)
model.fit(X, y, 1000, 0.01)

epoch 100 error = 6.5281
epoch 200 error = 6.2212
epoch 300 error = 5.9994
epoch 400 error = 5.8107
epoch 500 error = 5.6495
epoch 600 error = 5.5129
epoch 700 error = 5.399
epoch 800 error = 5.3052
epoch 900 error = 5.2291
epoch 1000 error = 5.1679


In [9]:
pred = model.predict(X)

In [10]:
print("Actual Age:", (y[:10] + 1.5).flatten())
print("Predicted Age:", (pred[:10] + 1.5).flatten())

Actual Age: [16.5  8.5 10.5 11.5  8.5  9.5 21.5 17.5 10.5 20.5]
Predicted Age: [10.17658608  9.30805143 12.32687467 11.15513758  8.0437351   9.18778778
 14.24016257 12.48090696 11.36571239 13.79333743]


In [11]:
mse = np.mean((y - pred) ** 2)
rmse = np.sqrt(mse)

print("\nMean Squared Error:", round(mse, 4))
print("Root Mean Squared Error:", round(rmse, 4))


Mean Squared Error: 5.1674
Root Mean Squared Error: 2.2732
